# Stroke Prediction — EDA & Model Training

## Dataset Description

**Source:** `healthcare-dataset-stroke-data.csv`

The dataset contains **5,110 patient records** used to predict whether a patient is likely to have a stroke. It includes demographic, lifestyle, and medical history features.

### Features

| Feature | Description | Type |
|---------|-------------|------|
| `id` | Unique identifier | int |
| `gender` | Male, Female, Other | categorical |
| `age` | Age of the patient | float |
| `hypertension` | 0 = No hypertension, 1 = Hypertension | binary |
| `heart_disease` | 0 = No heart disease, 1 = Heart disease | binary |
| `ever_married` | Yes / No | categorical |
| `work_type` | children, Govt_job, Never_worked, Private, Self-employed | categorical |
| `Residence_type` | Rural / Urban | categorical |
| `avg_glucose_level` | Average glucose level in blood | float |
| `bmi` | Body Mass Index | float |
| `smoking_status` | formerly smoked, never smoked, smokes, Unknown | categorical |
| **`stroke`** | **Target — 1 = had stroke, 0 = no stroke** | **binary** |

### Key Challenges
- **Severe class imbalance**: ~4.9% stroke cases vs ~95.1% non-stroke
- **Missing values**: `bmi` column has missing entries
- **Categorical encoding** required for several features

### Task
Binary classification — predict stroke occurrence. SMOTE is applied to handle imbalance.

---
## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve
)

sns.set_theme(style='whitegrid')
%matplotlib inline

DATA_DIR = os.path.join('..', 'data')
MODELS_DIR = os.path.join('..', 'models')

---
## 2. Load & Inspect the Data

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'healthcare-dataset-stroke-data.csv'))
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Missing values
print('Missing values per column:')
df.isnull().sum()

In [ ]:
# Unique values for categorical columns
for col in df.select_dtypes(include='object').columns:
    print(f'{col}: {df[col].unique()}')

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Target Distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x='stroke', data=df, ax=ax[0], palette='Set2')
ax[0].set_title('Stroke Distribution')
ax[0].set_xticklabels(['No Stroke (0)', 'Stroke (1)'])

df['stroke'].value_counts().plot.pie(autopct='%1.1f%%', labels=['No Stroke', 'Stroke'],
                                      colors=['#66c2a5','#fc8d62'], ax=ax[1])
ax[1].set_ylabel('')
ax[1].set_title('Target Class Proportion')

plt.tight_layout()
plt.show()

print(f'\nClass counts:\n{df["stroke"].value_counts()}')
print(f'\nStroke rate: {df["stroke"].mean()*100:.2f}%')

### 3.2 Age Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='age', hue='stroke', kde=True, ax=axes[0], palette='Set2', bins=30)
axes[0].set_title('Age Distribution by Stroke Status')

sns.boxplot(x='stroke', y='age', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('Age vs Stroke')
axes[1].set_xticklabels(['No Stroke', 'Stroke'])

plt.tight_layout()
plt.show()

### 3.3 BMI Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['bmi'].dropna(), bins=40, kde=True, ax=axes[0], color='coral')
axes[0].set_title('BMI Distribution')

sns.boxplot(x='stroke', y='bmi', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('BMI vs Stroke')
axes[1].set_xticklabels(['No Stroke', 'Stroke'])

plt.tight_layout()
plt.show()

### 3.4 Average Glucose Level

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='avg_glucose_level', hue='stroke', kde=True, ax=axes[0], palette='Set2', bins=40)
axes[0].set_title('Glucose Level Distribution by Stroke')

sns.boxplot(x='stroke', y='avg_glucose_level', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('Glucose Level vs Stroke')
axes[1].set_xticklabels(['No Stroke', 'Stroke'])

plt.tight_layout()
plt.show()

### 3.5 Categorical Features vs Stroke

In [ ]:
cat_cols = ['gender', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['stroke'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=axes[i], colormap='Set2')
    axes[i].set_title(f'{col} vs Stroke (%)')
    axes[i].set_ylabel('Percentage')
    axes[i].legend(title='Stroke', labels=['No', 'Yes'])
    axes[i].tick_params(axis='x', rotation=45)

# Hide unused subplots
for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

### 3.6 Correlation Heatmap (after encoding)

In [ ]:
df_encoded = df.drop(columns=['id']).copy()
df_encoded['bmi'] = df_encoded['bmi'].fillna(df_encoded['bmi'].median())

for col in df_encoded.select_dtypes(include='object').columns:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

plt.figure(figsize=(12, 8))
corr = df_encoded.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

---
## 4. Data Preprocessing

In [ ]:
df_proc = df.copy()
df_proc.drop(columns=['id'], inplace=True)

# Fill missing BMI with median
df_proc['bmi'] = df_proc['bmi'].fillna(df_proc['bmi'].median())
print(f'Remaining missing values: {df_proc.isnull().sum().sum()}')

# Encode categorical columns
categorical_cols = df_proc.select_dtypes(include='object').columns.tolist()
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_proc[col] = le.fit_transform(df_proc[col])
    label_encoders[col] = le
    print(f'{col}: {list(le.classes_)}')

df_proc.head()

In [ ]:
X = df_proc.drop('stroke', axis=1)
y = df_proc['stroke']

print(f'Features shape: {X.shape}')
print(f'Class distribution:\n{y.value_counts()}')

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE
print('Applying SMOTE...')
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print(f'After SMOTE — Class distribution:\n{pd.Series(y_train_resampled).value_counts()}')

# Save scaler and encoders
joblib.dump(scaler, os.path.join(MODELS_DIR, 'stroke_scaler.pkl'))
joblib.dump(label_encoders, os.path.join(MODELS_DIR, 'stroke_encoders.pkl'))

print(f'\nTrain set (resampled): {X_train_resampled.shape}')
print(f'Test set: {X_test_scaled.shape}')

---
## 5. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=7, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=7, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=7, random_state=42, eval_metric='logloss', verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=7, random_state=42, verbose=-1),
}

results = {}
for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results[name] = {'accuracy': acc, 'f1': f1, 'model': model}
    print(f'  Accuracy: {acc:.4f}  |  F1: {f1:.4f}')

In [ ]:
# Comparison bar chart
model_names = list(results.keys())
accuracies = [results[n]['accuracy'] for n in model_names]
f1_scores = [results[n]['f1'] for n in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#66c2a5')
bars2 = ax.bar(x + width/2, f1_scores, width, label='F1 Score', color='#fc8d62')

ax.set_ylabel('Score')
ax.set_title('Model Comparison — Stroke Prediction')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15)
ax.legend()
ax.set_ylim(0.0, 1.0)
ax.bar_label(bars1, fmt='%.3f', fontsize=8)
ax.bar_label(bars2, fmt='%.3f', fontsize=8)

plt.tight_layout()
plt.show()

---
## 6. Best Model — Detailed Evaluation

In [ ]:
best_name = max(results, key=lambda k: results[k]['accuracy'])
best_model = results[best_name]['model']
print(f'Best model: {best_name}')

y_pred = best_model.predict(X_test_scaled)
y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]

print(f"\nAccuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

In [ ]:
# Confusion Matrix & ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Stroke', 'Stroke'], yticklabels=['No Stroke', 'Stroke'])
axes[0].set_title(f'Confusion Matrix — {best_name}')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc_score(y_test, y_pred_proba):.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=True)

    plt.figure(figsize=(10, 6))
    feat_imp.plot(kind='barh', color='steelblue')
    plt.title(f'Feature Importance — {best_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

---
## 7. Save Best Model

In [ ]:
joblib.dump(best_model, os.path.join(MODELS_DIR, 'stroke_model.pkl'))
print(f'Best model ({best_name}) saved to models/stroke_model.pkl')
print(f'Scaler saved to models/stroke_scaler.pkl')
print(f'Label encoders saved to models/stroke_encoders.pkl')